# Shirt / T-shirt / blazer verification — Dima806 only

Select **GPU T4 ×2** and enable Internet. Attach `shirt_garment_verification_v1.zip` as an input dataset. This classifies images; it does not train a model or replace your project database.

Run setup, then the 300-image sample. The full-catalog cell is explicitly enabled when you choose. Save the result ZIP before ending the session.

In [ ]:
from pathlib import Path
import sys, os, subprocess, json, shutil, zipfile, hashlib
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
BASE=Path('/kaggle/input')
candidates=[p for p in BASE.rglob('classify_garments.py') if (p.parent/'garment_core.py').exists()]
if not candidates:
    archives=list(BASE.rglob('shirt_garment_verification_v1.zip'))
    assert len(archives)==1, f'Attach one supplied input ZIP: {archives}'
    unpack=Path('/kaggle/working/garment_code_v1')
    unpack.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        assert all((unpack/n).resolve().is_relative_to(unpack.resolve()) for n in z.namelist())
        z.extractall(unpack)
    candidates=list(unpack.rglob('classify_garments.py'))
assert len(candidates)==1, f'Attach one version of this package: {candidates}'
CODE=candidates[0].parent
SCRIPT=CODE/'classify_garments.py'
DATA=CODE/'input/shirts_master.parquet'
SAMPLE=CODE/'sample/sample_input.csv'
OUT=Path('/kaggle/working/garment_results_v1')
assert DATA.exists() and SAMPLE.exists(), 'The package must include input/ and sample/.'
manifest=json.loads((CODE/'package_manifest.json').read_text())
assert hashlib.sha256(DATA.read_bytes()).hexdigest()==manifest['master_sha256'], 'Input master checksum mismatch'
print('Code:',CODE,'\nMaster:',DATA,'\nSample:',SAMPLE,'\nResults:',OUT)
DEPS=Path('/kaggle/working/garment_python_v1')
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--upgrade','--target',str(DEPS),
                '-r',str(CODE/'requirements-kaggle.txt')],check=True)
os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+os.environ.get('PYTHONPATH','')
# Worker subprocesses use this isolated overlay. Kaggle's global Hub/Gradio and
# CUDA PyTorch installation are not downgraded or replaced.


## Optional restore — run before processing images
For a new session, attach the earlier output ZIP and set its path here. Leave None for your first run. Restore before running sample/full cells; never overwrite an existing checkpoint.

In [ ]:
RESTORE_ZIP=None  # Path('/kaggle/input/.../garment_results_backup.zip')
if RESTORE_ZIP:
    assert RESTORE_ZIP.exists()
    assert not (OUT/'checkpoint.sqlite').exists(), 'Output already has a checkpoint; do not overwrite it.'
    with zipfile.ZipFile(RESTORE_ZIP) as z:
        assert all((OUT.parent/n).resolve().is_relative_to(OUT.parent.resolve()) for n in z.namelist())
        z.extractall(OUT.parent)
    assert (OUT/'checkpoint.sqlite').exists(), 'Backup output folder name differs; select its OUT path.'


## Confirm hardware and run package checks
Do not install a replacement PyTorch wheel. Kaggle supplies CUDA-enabled PyTorch.

In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()>=2, 'Select GPU T4 x2 in Kaggle Settings.'
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(CODE/'tests'),'-v'],check=True)


## Run the varied sample
`LIMIT=0` means all remaining sample images. It does **not** mean the full catalog. Scores/margins are heuristic gates, not measured accuracy. Shirt is the only accepted target; T-shirt and Polo remain separate classes.

In [ ]:
LIMIT=0
MIN_SCORE=0.85
MIN_MARGIN=0.20
COMMON=['--output-dir',str(OUT),'--devices','0,1','--batch-size','24',
        '--download-workers','12','--min-score',str(MIN_SCORE),'--min-margin',str(MIN_MARGIN)]
subprocess.run([sys.executable,'-u',str(SCRIPT),'run','--input',str(SAMPLE),
                '--limit-images',str(LIMIT),'--preview-count','300',*COMMON],check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage
summary=json.loads((OUT/'summary.json').read_text())
display(summary)
results=pd.read_parquet(OUT/'products_annotated.parquet')
display(results[['parent_asin','title','sleeve','pattern','garment_type','garment_score','garment_margin','garment_status']].head(30))
for path in sorted((OUT/'previews').glob('*.jpg'))[:12]:
    print(path.stem)
    display(DisplayImage(filename=str(path)))
speed=summary.get('images_per_second')
if speed:
    remaining=max(0,manifest['unique_valid_urls']-summary['processed_unique_urls'])
    print(f'Illustrative remaining processing time at this measured sample rate: {remaining/speed/3600:.2f} h')
    print('Startup/download overhead and network variation affect this estimate.')


## Optional measured evaluation
Download `reference_labels.csv`, fill a small verified set using the exact category names in `label_names.json`, then attach that CSV as a new input. The script reports shirt retention and non-shirt false acceptance. Leave this cell disabled if no reference labels have been supplied; do not interpret model scores as accuracy.

In [ ]:
RUN_EVALUATION=False
LABELS=None  # Set Path('/kaggle/input/.../your_verified_labels.csv') if available.
if RUN_EVALUATION:
    assert LABELS and LABELS.exists(), 'Select your manually verified label CSV.'
    subprocess.run([sys.executable,str(SCRIPT),'evaluate','--labels',str(LABELS),
                    '--predictions',str(OUT/'products_annotated.parquet'),
                    '--output',str(OUT/'evaluation.json')],check=True)
    display(json.loads((OUT/'evaluation.json').read_text()))


## Full catalog — reuse the sample checkpoint
Set `RUN_FULL_DATASET=True` when ready. The input is the original 50,224-row master, processed once per unique URL. Existing sample predictions resume automatically. No model training is performed.

For a resumed session, use the restore cell near the top before processing. You can then skip the sample and run this full-catalog cell.

In [ ]:
RUN_FULL_DATASET=False
if RUN_FULL_DATASET:
    subprocess.run([sys.executable,'-u',str(SCRIPT),'run','--input',str(DATA),
                    '--preview-count','300',*COMMON],check=True)
    display(json.loads((OUT/'summary.json').read_text()))


## Save results
Run after the worker exits. Download the entire ZIP, including the checkpoint and previews. `products_annotated.parquet` retains all input rows; the three candidate files partition those rows. None replaces your live database automatically. Failed/pending images stay in the uncertain output.

In [ ]:
from IPython.display import FileLink
assert OUT.exists()
archive=shutil.make_archive('/kaggle/working/garment_results_backup','zip',OUT.parent,OUT.name)
display(FileLink(archive))
